# Sample Material Generation Code

In [1]:
import numpy as np
import pandas
import matplotlib.pyplot as plt
from scipy import interpolate
import mitsuba as mi
import os

mi.set_variant('scalar_spectral')

#set wavelength range, increments by 10
nm = np.arange(300,801,10)

outputDir = "C:/Users/marin/ssynth_new/data/Materials/"
os.makedirs(outputDir,exist_ok=True)

# Creation of Epidermis Material

In [10]:
# Reduced scattering coefficient of epidermis (inverse cm)
musEpi = 117*(nm/500)**(-1.6)
adict = {'wavelength':nm, 'rScattering':musEpi}
df_musEpi = pandas.DataFrame(adict)
df_musEpi.to_csv(outputDir + '/epidermis_rScattering.spd', sep = ' ', header=False, index=False)

#refractive index for epidermis between 1.42-1.44
iorEpi = 1.43

#melanosome absorption coefficient (inverse cm)
muaMel = 6.6e11 * nm**(-3.33)

melanosomes = [float(x) / 100 for x in range(1, 51)] #Melanosome fraction 0.01-0.50
muaEpi = {}
df_muaEpi = {}
for mel in melanosomes:
    #absorption coefficient of epidermis (inverse cm)
    muaEpi[mel] = mel * muaMel + (1-mel)*(0.244 + 85.3 * np.exp(-(nm-154)/66.2))
    adict = {'wavelength':nm, 'absorption':muaEpi[mel]}
    df_muaEpi[mel] = pandas.DataFrame(adict)
    df_muaEpi[mel].to_csv(outputDir + '/epidermis_abs_mel' + str(mel) + '.spd', sep = ' ', header=False, index=False)
    
    #extinction coefficient = absorption + scattering
    extEpi = muaEpi[mel] + musEpi
    adict = {'wavelength':nm, 'extinction':extEpi}
    df_extEpi = pandas.DataFrame(adict)
    df_extEpi.to_csv(outputDir + '/epidermis_ext_mel' + str(mel) + '.spd', sep = ' ', header=False, index=False)
    
    #albedo = scattering / extinction coefficient
    albEpi = musEpi/extEpi
    adict = {'wavelength':nm, 'albedo':albEpi}
    df_albEpi = pandas.DataFrame(adict)
    df_albEpi.to_csv(outputDir + '/epidermis_alb_mel' + str(mel) + '.spd', sep = ' ', header=False, index=False)

# Creation of Blood Material

In [2]:
#import tabulated molar extinction coefficients for deoxy-hemoglobin in water for 300-800nm in 10nm increments
df_molarExtHb = pandas.read_csv (outputDir + 'Hb_molarExtinction_valuesOnly.spd', header = None)
molarExtHb = np.squeeze(np.array(df_molarExtHb))
#import tabulated molar extinction coefficients for oxy-hemoglobin in water for 300-800nm in 10nm increments
df_molarExtHbO2 = pandas.read_csv (outputDir + 'HbO2_molarExtinction_valuesOnly.spd', header = None)
molarExtHbO2 = np.squeeze(np.array(df_molarExtHbO2))

#absorption coefficient of whole blood assuming Hb concentration = 150g/liter
#64500g/mole is the gram molecular weight of Hb
muaHb = 2.303 * molarExtHb * 150 / 64500
adict = {'wavelength':nm, 'absorption':muaHb}
df_muaHb = pandas.DataFrame(adict)
df_muaHb.to_csv(outputDir + '/blood_Hb_absorption' + '.spd', sep = ' ', header=False, index=False)

muaHbO2 = 2.303 * molarExtHbO2 * 150 / 64500
adict = {'wavelength':nm, 'absorption':muaHbO2}
df_muaHbO2 = pandas.DataFrame(adict)
df_muaHbO2.to_csv(outputDir + '/blood_HbO2_absorption' + '.spd', sep = ' ', header=False, index=False)

#reduced scattering coefficient of whole blood (inverse cm)
musBlood = 22*(nm/500)**(-0.66)
adict = {'wavelength':nm, 'absorption':musBlood}
df_musBlood = pandas.DataFrame(adict)
df_musBlood.to_csv(outputDir + '/blood_rScattering' + '.spd', sep = ' ', header=False, index=False)

#extinction coefficient = absorption + scattering
extHb = muaHb + musBlood
adict = {'wavelength':nm, 'extinction':extHb}
df_extHb = pandas.DataFrame(adict)
df_extHb.to_csv(outputDir + '/blood_Hb_ext' + '.spd', sep = ' ', header=False, index=False)

extHbO2 = muaHbO2 + musBlood
adict = {'wavelength':nm, 'extinction':extHbO2}
df_extHbO2 = pandas.DataFrame(adict)
df_extHbO2.to_csv(outputDir + '/blood_HbO2_ext' + '.spd', sep = ' ', header=False, index=False)

#albedo = scattering / extinction coefficient
albHb = musBlood/extHb
adict = {'wavelength':nm, 'albedo':albHb}
df_albHb = pandas.DataFrame(adict)
df_albHb.to_csv(outputDir + '/blood_Hb_alb' + '.spd', sep = ' ', header=False, index=False)

albHbO2 = musBlood/extHbO2
adict = {'wavelength':nm, 'albedo':albHbO2}
df_albHbO2 = pandas.DataFrame(adict)
df_albHbO2.to_csv(outputDir + '/blood_HbO2_alb' + '.spd', sep = ' ', header=False, index=False)
    
#refractive index for blood = 1.36 for 680-930nm
iorBlood = 1.36

# Creation of Papillary Material 

In [ ]:
#reduced scattering coefficient of dermis (inverse cm)
muspapillary = 66*(nm/500)**(-0.7)
adict = {'wavelength':nm, 'rScattering':muspapillary}
df_muspapillary = pandas.DataFrame(adict)
df_muspapillary.to_csv(outputDir + '/papillary_rScattering.spd', sep = ' ', header=False, index=False)

#refractive index for dermis is wavelength-dependent, but cannot input spectrum for ior in bsdf
#therefore, will normalize to lambda = 500nm
A = 1.3696
B = 3916.8
C = 2558.8
iorpapillary = A + (B/(500**2)) + (C/(500**4))

#fB = range 2-5%
fractionBlood = [0.011, 0.015, 0.02, 0.025]
muapapillary = {}
df_muapapillary = {}
for fB in fractionBlood:
    #absorption coefficient of dermis (inverse cm)
    muapapillary[fB] = fB * muaHbO2 + (1-fB)*(0.244 + 85.3 * np.exp(-(nm-154)/66.2))
    adict = {'wavelength':nm, 'absorption':muapapillary[fB]}
    df_muapapillary[fB] = pandas.DataFrame(adict)
    df_muapapillary[fB].to_csv(outputDir + '/papillary_abs_fB' + str(fB) + '.spd', sep = ' ', header=False, index=False)
    
    #extinction coefficient = absorption + scattering
    extpapillary = muapapillary[fB] + muspapillary
    adict = {'wavelength':nm, 'extinction':extpapillary}
    df_extpapillary = pandas.DataFrame(adict)
    df_extpapillary.to_csv(outputDir + '/papillary_ext_fB' + str(fB) + '.spd', sep = ' ', header=False, index=False)
    
    #albedo = scattering / extinction coefficient
    albpapillary = muspapillary/extpapillary
    adict = {'wavelength':nm, 'albedo':albpapillary}
    df_albpapillary = pandas.DataFrame(adict)
    df_albpapillary.to_csv(outputDir + '/papillary_alb_fB' + str(fB) + '.spd', sep = ' ', header=False, index=False)

# Creation of Upper Blood Material 

In [3]:
#reduced scattering coefficient of dermis (inverse cm)
musupperblood = 66*(nm/500)**(-0.7)
adict = {'wavelength':nm, 'rScattering':musupperblood}
df_musupperblood = pandas.DataFrame(adict)
df_musupperblood.to_csv(outputDir + '/upper_blood_rScattering.spd', sep = ' ', header=False, index=False)

#refractive index for dermis is wavelength-dependent, but cannot input spectrum for ior in bsdf
#therefore, will normalize to lambda = 500nm
A = 1.3696
B = 3916.8
C = 2558.8
iorupperblood = A + (B/(500**2)) + (C/(500**4))


fractionBlood = [0.1,0.2, 0.3, 0.4, 0.5] #fB = 
muaupperblood = {}
df_muaupperblood = {}
for fB in fractionBlood:
    #absorption coefficient of dermis (inverse cm)
    muaupperblood[fB] = fB * muaHbO2 + (1-fB)*(0.244 + 85.3 * np.exp(-(nm-154)/66.2))
    adict = {'wavelength':nm, 'absorption':muaupperblood[fB]}
    df_muaupperblood[fB] = pandas.DataFrame(adict)
    df_muaupperblood[fB].to_csv(outputDir + '/upper_blood_abs_fB' + str(fB) + '.spd', sep = ' ', header=False, index=False)
    
    #extinction coefficient = absorption + scattering
    extupperblood = muaupperblood[fB] + musupperblood
    adict = {'wavelength':nm, 'extinction':extupperblood}
    df_extupperblood = pandas.DataFrame(adict)
    df_extupperblood.to_csv(outputDir + '/upper_blood_ext_fB' + str(fB) + '.spd', sep = ' ', header=False, index=False)
    
    #albedo = scattering / extinction coefficient
    albupperblood = musupperblood/extupperblood
    adict = {'wavelength':nm, 'albedo':albupperblood}
    df_albupperblood = pandas.DataFrame(adict)
    df_albupperblood.to_csv(outputDir + '/upper_blood_alb_fB' + str(fB) + '.spd', sep = ' ', header=False, index=False)

# Creation of Reticular Material 

In [6]:
#reduced scattering coefficient of dermis (inverse cm)
musreticular = 66*(nm/500)**(-0.7)
adict = {'wavelength':nm, 'rScattering':musreticular}
df_musreticular = pandas.DataFrame(adict)
df_musreticular.to_csv(outputDir + '/reticular_rScattering.spd', sep = ' ', header=False, index=False)

#refractive index for dermis is wavelength-dependent, but cannot input spectrum for ior in bsdf
#therefore, will normalize to lambda = 500nm
A = 1.3696
B = 3916.8
C = 2558.8
iorreticular = A + (B/(500**2)) + (C/(500**4))

#fB = volume fraction of blood; range = 0.2-7%
#fB = typical 0.2%; concentrated in venous plexus 2-5%
fractionBlood = [0.0075, 0.01, 0.012, 0.014]
muareticular = {}
df_muareticular = {}
for fB in fractionBlood:
    #absorption coefficient of dermis (inverse cm)
    muareticular[fB] = fB * muaHbO2 + (1-fB)*(0.244 + 85.3 * np.exp(-(nm-154)/66.2))
    adict = {'wavelength':nm, 'absorption':muareticular[fB]}
    df_muareticular[fB] = pandas.DataFrame(adict)
    df_muareticular[fB].to_csv(outputDir + '/reticular_abs_fB' + str(fB) + '.spd', sep = ' ', header=False, index=False)
    
    #extinction coefficient = absorption + scattering
    extreticular = muareticular[fB] + musreticular
    adict = {'wavelength':nm, 'extinction':extreticular}
    df_extreticular = pandas.DataFrame(adict)
    df_extreticular.to_csv(outputDir + '/reticular_ext_fB' + str(fB) + '.spd', sep = ' ', header=False, index=False)
    
    #albedo = scattering / extinction coefficient
    albreticular = musreticular/extreticular
    adict = {'wavelength':nm, 'albedo':albreticular}
    df_albreticular = pandas.DataFrame(adict)
    df_albreticular.to_csv(outputDir + '/reticular_alb_fB' + str(fB) + '.spd', sep = ' ', header=False, index=False)

# Creation of Deep Blood Material 

In [4]:
#reduced scattering coefficient of dermis (inverse cm)
musdeepblood = 66*(nm/500)**(-0.7)
adict = {'wavelength':nm, 'rScattering':musdeepblood}
df_musdeepblood = pandas.DataFrame(adict)
df_musdeepblood.to_csv(outputDir + '/deep_blood_rScattering.spd', sep = ' ', header=False, index=False)

#refractive index for dermis is wavelength-dependent, but cannot input spectrum for ior in bsdf
#therefore, will normalize to lambda = 500nm
A = 1.3696
B = 3916.8
C = 2558.8
iordeepblood = A + (B/(500**2)) + (C/(500**4))

#fB = volume fraction of blood; range = 0.2-7%
#fB = typical 0.2%; concentrated in venous plexus 2-5%
fractionBlood = [0.1, 0.2, 0.3, 0.4, 0.5]
muadeepblood = {}
df_muadeepblood = {}
for fB in fractionBlood:
    #absorption coefficient of dermis (inverse cm)
    muadeepblood[fB] = fB * muaHbO2 + (1-fB)*(0.244 + 85.3 * np.exp(-(nm-154)/66.2))
    adict = {'wavelength':nm, 'absorption':muadeepblood[fB]}
    df_muadeepblood[fB] = pandas.DataFrame(adict)
    df_muadeepblood[fB].to_csv(outputDir + '/deep_blood_abs_fB' + str(fB) + '.spd', sep = ' ', header=False, index=False)
    
    #extinction coefficient = absorption + scattering
    extdeepblood = muadeepblood[fB] + musdeepblood
    adict = {'wavelength':nm, 'extinction':extdeepblood}
    df_extdeepblood = pandas.DataFrame(adict)
    df_extdeepblood.to_csv(outputDir + '/deep_blood_ext_fB' + str(fB) + '.spd', sep = ' ', header=False, index=False)
    
    #albedo = scattering / extinction coefficient
    albdeepblood = musdeepblood/extdeepblood
    adict = {'wavelength':nm, 'albedo':albdeepblood}
    df_albdeepblood = pandas.DataFrame(adict)
    df_albdeepblood.to_csv(outputDir + '/deep_blood_alb_fB' + str(fB) + '.spd', sep = ' ', header=False, index=False)

# Creation of Hypodermis Material

In [5]:
#absorption coefficient of hypodermis (inverse cm)
muaHypo = ((-1/600)*nm) + (143/60)
adict = {'wavelength':nm, 'absorption':muaHypo}
df_muaHypo = pandas.DataFrame(adict)
df_muaHypo.to_csv(outputDir + '/hypo_absorption' + '.spd', sep = ' ', header=False, index=False)

#reduced scattering coefficient of hypodermis (inverse cm)
musHypo = ((-11/600)*nm) + (1687/60)
adict = {'wavelength':nm, 'absorption':musHypo}
df_musHypo = pandas.DataFrame(adict)
df_musHypo.to_csv(outputDir + '/hypo_rScattering' + '.spd', sep = ' ', header=False, index=False)

#extinction coefficient = absorption + scattering
extHypo = muaHypo + musHypo
adict = {'wavelength':nm, 'extinction':extHypo}
df_extHypo = pandas.DataFrame(adict)
df_extHypo.to_csv(outputDir + '/hypo_ext' + '.spd', sep = ' ', header=False, index=False)

#albedo = scattering / extinction coefficient
albHypo = musHypo/extHypo
adict = {'wavelength':nm, 'albedo':albHypo}
df_albHypo = pandas.DataFrame(adict)
df_albHypo.to_csv(outputDir + '/hypo_alb' + '.spd', sep = ' ', header=False, index=False)

#refractive index for hypodermis
iorHypo = 1.44